# 📗 SLO 정의와 원인 분석, 개선: 좋은 요청 비율로 목표를 정하고 배포가 만든 장애를 찾아 고치기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">이 교안을 배우는 이유</strong></p>
<p style="margin:6px 0;">응답 시간과 오류 수를 모아도 어느 정도면 문제인지 기준이 없으면 고칠지 판단하기 어렵습니다. 이 교안에서는 지연·품질·비용의 목표를 정해 배포 전후를 같은 기준으로 판정하고, 원인을 고친 뒤 회복했는지 확인합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/29_why_slo.png" width="1050" alt="15초 안에 답한 요청이 90퍼센트인 결과를 목표 95퍼센트 이상과 비교해 개선이 필요하다고 판단하는 예시입니다.">

#### **SLO(Service Level Objective, 서비스 수준 목표)**

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서비스가 어느 기간 동안 어느 수준을 지켜야 하는지 정한 목표</strong>입니다. 예를 들어 &quot;4주 측정 창에서 유효 요청 100건 중 99건 이상이 5xx가 아니어야 한다&quot;처럼 정합니다. 그림의 SLI는 실제로 목표를 지킨 요청의 비율, 측정 창은 그 비율을 계산할 기간입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">활용처:</strong> 배포 뒤 서비스가 나빠졌는지 판정하기, 경보 기준 정하기, 원인을 고친 뒤 회복했는지 확인하기입니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/30_slo_concept.png" width="1050" alt="유효 요청 100건 중 좋은 요청 95건과 나쁜 요청 5건을 보여 줍니다. 목표 95%와 에러 버짓 5%를 막대 비율로 비교하고, 401·403·422·429는 유효 요청에서 제외합니다.">

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">

<span class="day79-goal" style="font-size:1.13em;color:#1f3a5f;">**학습 목표**</span>

1. 서빙 API의 SLI를 좋은 요청 / 유효 요청으로 정의하고, SLO 테이블로 측정 창 하나를 판정합니다.
2. 에러 버짓과 소진 속도(burn rate)를 계산해 목표보다 빠르게 버짓을 쓰는지 판단합니다.
3. 배포가 만든 장애를 배포 버전별 trace와 vLLM 메트릭으로 비교해 원인을 좁힙니다.
4. 되돌리기와 고친 커밋으로 개선하고, 같은 부하와 같은 테이블로 네 창을 비교합니다.

</div>

## 1. 실습 준비

### 1-1. 노트북 준비

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01, 02의 준비 코드를 한 셀에 모았습니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 앞 교안에서 만든 연결·부하·메트릭·trace 도구를 이어 사용해 같은 측정 절차를 반복합니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# ---- 라이브러리, 경로, 설정값(교안 01, 02와 같습니다) ----
import json
import os
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv

day_dir = Path("..").resolve()       # data, images, output 이 있는 일차 폴더(day79_서빙운영_관측_SLO)
data_dir = day_dir / "data"
output_dir = day_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# 키와 주소는 실습자료/.env 에 모아 둡니다(화면에 출력하지 않습니다)
env_path = day_dir.parent / ".env"
load_dotenv(env_path, override=True)

# 테이블의 긴 글(질문, 답)이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# ---- JSONL 읽기와 질문 ----
def load_jsonl(path):
    """path가 가리키는 UTF-8 JSONL 파일을 읽어 딕셔너리 목록으로 돌려줍니다.

    파일의 각 줄을 JSON 딕셔너리 하나로 바꾸고, 원래 줄 순서를 유지해 리스트에 담습니다.
    질문 파일은 보통 한 딕셔너리에 question과 기대하는 메타데이터를 담습니다.
    """
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


labor_questions = load_jsonl(data_dir / "본실습" / "labor_questions.jsonl")
lease_questions = load_jsonl(data_dir / "따라하기" / "lease_questions.jsonl")

# ---- 주소와 헤더 ----
# VM 의 서빙 API 주소와 관리자 키 헤더
VM_URL = "http://" + os.environ["VM_HOST"] + ":8080"
ADMIN_HEADERS = {"X-Admin-Key": os.environ["ADMIN_API_KEY"]}

# 두 층의 /metrics 주소. 서빙 API 는 관리자 키가 있어야 열리고, vLLM 은 키 없이 열린다(/v1 로 시작하지 않는 경로)
SERVING_METRICS_URL = VM_URL + "/metrics"
VLLM_METRICS_URL = "https://" + os.environ["GPU_POD_ID"] + "-8000.proxy.runpod.net/metrics"

# 노트북이 VM 의 서빙 API 응답을 기다리는 최대 시간(초). VM 의 vLLM 시간 제한 90초보다 길게 둡니다
REQUEST_TIMEOUT = 120

# 시간당 요금(달러). 교안 01 1-1에서 .env 에 적은 Total / hour 값입니다
GPU_PRICE_PER_HOUR = float(os.environ["GPU_PRICE_PER_HOUR"])

# ---- GitHub 실행 기록 ----
# GitHub API 는 로그인 없이 부르면 IP 하나에 시간당 60번이라 같은 와이파이의 교실 전체가 나눠 씁니다. 읽기 전용 토큰을 붙이면 학생마다 시간당 5000번입니다
GITHUB_HEADERS = {"Authorization": "Bearer " + os.environ["GITHUB_READ_TOKEN"], "Accept": "application/vnd.github+json"}  # 배포 단원 교안 00 5-2 의 읽기 전용 토큰(값은 출력하지 않습니다)

# 이 저장소의 GitHub REST API 주소(실행 목록은 최대 60초 늦게 바뀔 수 있습니다)
GITHUB_API = "https://api.github.com/repos/" + os.environ["GITHUB_OWNER"] + "/serving-deploy"


def recent_runs():
    """GitHub Actions의 최근 워크플로 실행 20개를 읽어 표로 돌려줍니다.

    ci, cd, rollback 실행을 함께 읽습니다. 각 행은 실행 한 건이며, 이름, 제목, 커밋 sha,
    시작 이벤트, 상태, 결론, 실행 링크를 담습니다. GitHub가 돌려준 최신 실행이 표의 위쪽에 옵니다.
    """
    response = httpx.get(GITHUB_API + "/actions/runs", params={"per_page": 20}, headers=GITHUB_HEADERS, timeout=30)
    response.raise_for_status()
    runs = pd.DataFrame(response.json()["workflow_runs"])
    return runs[["id", "name", "display_title", "head_sha", "event", "status", "conclusion", "html_url"]]


# 저장소 화면 주소(GitHub 화면을 열 때 씁니다)
REPO_URL = "https://github.com/" + os.environ["GITHUB_OWNER"] + "/serving-deploy"


# ---- 부하 보내기 ----
import asyncio
import time

# 동시에 보낼 요청 수(서빙 벤치마킹 단원의 기본 부하 16)
MAX_CONCURRENCY = 16
semaphore = asyncio.Semaphore(MAX_CONCURRENCY)


async def run_limited(task):
    """동시에 실행할 요청 수를 제한하고, 한 요청이 끝나면 그 결과를 돌려줍니다.

    task에는 실행할 비동기 요청 함수가 들어옵니다. semaphore 자리가 날 때까지 기다린 뒤
    task를 실행하고, 요청 함수가 돌려준 결과를 반환합니다. 요청 오류는 여기서 감추지 않고 그대로 전달합니다.
    """
    async with semaphore:
        return await task


async def send_chat(client, path, body):
    """서비스 키가 설정된 비동기 client로 질문을 보내고 요청 결과 한 행을 돌려줍니다.

    path는 /v1/labor/chat 같은 상담 주소이고, body는 question을 담은 딕셔너리입니다.
    반환 딕셔너리에는 HTTP 상태 코드와 내 PC에서 잰 걸린 시간이 들어갑니다.
    상태 코드가 200이면 답의 끝난 이유, 출력 토큰 수, 답 글자 수도 넣습니다.
    200이 아니면 서버가 돌려준 오류 코드를 넣어, 429·502·504 응답도 다음 질문을 계속 보낼 수 있습니다.
    네트워크 연결 자체가 끊기거나 응답을 JSON으로 읽지 못한 오류는 그대로 전달됩니다.
    """
    started = time.perf_counter()
    response = await client.post(path, json=body)
    seconds = time.perf_counter() - started

    reply = response.json()
    row = {"상태 코드": response.status_code, "걸린 시간(초)": round(seconds, 2)}
    if response.status_code == 200:
        row["끝난 이유"] = reply["finish_reason"]
        row["출력 토큰"] = reply["usage"]["completion_tokens"]
        # 생각이 생성 상한을 다 쓰면 답(answer)이 None 으로 온다. 그때 답 글자 수는 0
        row["답 글자 수"] = 0
        if reply["answer"] is not None:
            row["답 글자 수"] = len(reply["answer"])
    else:
        row["오류"] = reply["error"]["code"]
    return row


# ---- /metrics 읽기 ----
# prometheus_client가 제공하는 해석 함수입니다. 숫자가 적힌 문자열을 이름·라벨·값으로 나눕니다.
from prometheus_client.parser import text_string_to_metric_families


def read_metrics(url, headers=None):
    """url의 메트릭 조회 주소에 요청해 서버가 기록한 숫자를 표로 돌려줍니다.

    headers에는 관리자 키처럼 조회 권한에 필요한 HTTP 헤더를 넣습니다. 서버 응답은 JSON이 아니라
    Prometheus 형식의 텍스트이며, text_string_to_metric_families가 이를 메트릭 묶음(family)과
    숫자가 적힌 줄(sample)로 나눕니다. 반환 표는 숫자 줄마다 한 행이며 name(이름),
    labels(구분 라벨), value(숫자) 열을 가집니다. HTTP 오류가 오면 예외를 내고 멈춥니다.
    """
    # 서버의 조회 주소에 GET 요청을 보냅니다. 응답 본문은 JSON이 아니라 숫자가 적힌 문자열입니다.
    response = httpx.get(url, headers=headers, timeout=30)
    response.raise_for_status()  # 서버가 정상 응답하지 않았으면 여기서 오류를 알립니다.

    # 해석 함수가 같은 메트릭에 속한 줄을 family로 묶고, 각 숫자 줄을 sample로 제공합니다.
    rows = []
    for family in text_string_to_metric_families(response.text):
        for sample in family.samples:
            # 예: name=요청 수 메트릭 이름, labels={"status": "200"}, value=3.0
            rows.append({"name": sample.name, "labels": sample.labels, "value": sample.value})
    return pd.DataFrame(rows)  # 숫자 줄 하나를 표의 한 행으로 만듭니다.


def metric_sum(snapshot, name, **labels):
    """스냅샷 표에서 메트릭 이름과 라벨 조건이 맞는 숫자를 모두 더해 돌려줍니다.

    name에는 serving_requests_total처럼 메트릭 이름을 넣습니다. status="200" 또는
    path="/v1/labor/chat"처럼 필요한 라벨을 더 줄 수 있습니다. 라벨을 여러 개 주면
    모두 같은 행만 더하며, 조건에 맞는 행이 없으면 0을 돌려줍니다.
    """
    # **labels에는 status="200", path="/v1/labor/chat"처럼 넘긴 조건이 딕셔너리로 모입니다.
    total = 0.0
    for row in snapshot.itertuples():
        if row.name != name:  # 다른 메트릭은 건너뜁니다.
            continue
        # 요청한 라벨 조건이 모두 맞는지 확인합니다. 조건을 주지 않았다면 같은 이름의 행을 모두 더합니다.
        matched = True
        for key, value in labels.items():
            if row.labels.get(key) != value:
                matched = False
        if matched:
            total += row.value
    return total  # 맞는 행이 없으면 처음 값인 0입니다.


# ---- Langfuse 클라이언트와 기록 읽기 ----
from langfuse import get_client

# .env 의 LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_BASE_URL 을 읽습니다(에이전트 관측 단원과 같은 방법)
langfuse = get_client()


def fetch_generations(start, end):
    """UTC 시각 start부터 end까지 시작한 모델 호출(generation) 기록을 표로 돌려줍니다.

    Langfuse API를 100건씩 페이지를 넘겨 가며 끝까지 읽습니다. 반환 표에서 모델 호출 하나가 한 행이며,
    trace id, 시작 시각, 사용자 id, 배포 버전, 창구 태그, 오류 레벨, 지연(초), 출력 토큰,
    비용(달러)을 확인할 수 있습니다. 실패해 토큰 사용량이 없는 호출은 출력 토큰이 비어 있습니다.
    """
    rows = []
    cursor = None
    while True:
        # fields: 기본 칸(basic) 에 사용량(usage), 지연(metrics), 태그(trace_context)를 더해 받습니다. 한 번에 100개씩
        page = langfuse.api.observations.get_many(
            type="GENERATION",
            from_start_time=start,
            to_start_time=end,
            fields="core,basic,usage,metrics,trace_context",
            limit=100,
            cursor=cursor,
        )
        for observation in page.data:
            # 실패한 호출은 사용량이 비어 있습니다
            output_tokens = None
            if observation.usage_details is not None:
                output_tokens = observation.usage_details.get("output")
            rows.append({
                "trace_id": observation.trace_id,
                "시작 시각": observation.start_time,
                "user_id": observation.user_id,
                "version": observation.version,
                "tags": observation.tags,
                "level": observation.level,
                "지연(초)": observation.latency,
                "출력 토큰": output_tokens,
                "비용(달러)": observation.total_cost,
            })

        # 다음 쪽이 없으면 cursor 가 None 입니다
        cursor = page.meta.cursor
        if cursor is None:
            break

    # 기록이 아직 하나도 없어도 열 이름은 있게 만든다(빈 테이블에서 열을 고르다 멈추지 않게)
    columns = ["trace_id", "시작 시각", "user_id", "version", "tags", "level", "지연(초)", "출력 토큰", "비용(달러)"]
    return pd.DataFrame(rows, columns=columns)

### 1-2. 사용자 키 발급

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 02와 같이 이 노트북에서 쓸 키를 새로 발급합니다. 노무 부하는 교안 01, 02의 pro 사용자 셋에, 전월세 부하(3절과 5절에서 함께 측정해 8절 함께 따라하기에서 씁니다)는 pro 사용자 둘에 나눠 보냅니다. 같은 키로 노무와 전월세를 1분 안에 연달아 보내면 60초에 30번 한도를 넘으므로 키를 따로 둡니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적·활용:</strong> 사용자 키를 나눠 요청과 trace를 사용자별로 구분하고, 전월세 창은 뒤의 따라하기에서 재사용합니다.</p>
</div>

In [ ]:
from datetime import datetime, timezone

# 노무 부하용 셋, 전월세 부하용 둘에 pro 키를 발급합니다(키 원문은 변수에만 담고 출력하지 않습니다)
issued_keys = {}
user_clients = {}
issued_rows = []
# 노무 부하용 사용자 셋(교안 01, 02 와 같은 이름)과 전월세 부하용 사용자 둘
labor_users = ["labor-app", "hr-team", "mobile-app"]
lease_users = ["lease-app", "lease-web"]
for user in labor_users + lease_users:
    issue_response = httpx.post(VM_URL + "/v1/admin/keys", headers=ADMIN_HEADERS, json={"user": user, "plan": "pro"}, timeout=REQUEST_TIMEOUT)
    issued = issue_response.json()
    issued_keys[user] = issued
    issued_rows.append({"상태 코드": issue_response.status_code, "key_id": issued["key_id"], "user": issued["user"], "plan": issued["plan"]})
    # 사용자마다 base_url 과 서비스 키 헤더를 담은 AsyncClient
    user_clients[user] = httpx.AsyncClient(base_url=VM_URL, headers={"X-API-Key": issued["api_key"]}, timeout=REQUEST_TIMEOUT)

# 부하를 나눠 보낼 클라이언트 목록
labor_clients = []
for user in labor_users:
    labor_clients.append(user_clients[user])
lease_clients = []
for user in lease_users:
    lease_clients.append(user_clients[user])
display(pd.DataFrame(issued_rows))

### 1-3. 측정 창 함수

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 교안은 같은 부하를 여섯 번(노무 네 번, 전월세 두 번) 보내고 매번 같은 계산을 합니다. 그래서 교안 01 4절에서 셀로 나눠 한 일(데움 요청, 부하 전 두 스냅샷, 부하, 부하 뒤 두 스냅샷)을 함수 하나로 묶습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 창(measurement window)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">SLI(2절에서 정의하는 좋은 요청의 비율)를 계산하는 시간 범위입니다. 이 교안에서는 부하 한 번의 시작부터 끝까지가 창 하나입니다.</li>
<li style="margin:3px 0;">예를 들어 노무 질문 64건을 보내기 시작한 때부터 마지막 응답을 받은 때까지가 창 하나입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code>의 카운터는 서버가 켜진 뒤로 계속 쌓이므로, 창의 값은 부하 뒤 스냅샷에서 부하 전 스냅샷을 뺀 차이(교안 01 4-3)로 계산합니다.</li>
</ul>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> API·vLLM 메트릭의 전후 차이와 요청별 응답을 같은 부하에 묶어, 다음 SLI·원가 비교에 씁니다.</p>
</div>

In [ ]:
async def measure_window(label, path, questions, clients):
    """질문 한 묶음을 보내고 SLI 계산에 필요한 결과를 한 측정 창으로 돌려줍니다.

    label은 창을 구분할 이름, path는 상담 주소, questions는 보낼 질문 목록,
    clients는 각각 다른 사용자 키가 담긴 비동기 클라이언트 목록입니다.
    함수는 먼저 질문 한 건으로 모델을 데운 다음, 부하 전 스냅샷을 읽고 질문을 키에 번갈아 나눠 보냅니다.
    반환 딕셔너리에는 요청별 결과, UTC 시작·끝 시각, 서빙 API와 vLLM의 부하 전후 스냅샷이 들어 있습니다.
    """
    # 데움 요청: 첫 사용자 키로 첫 질문 한 건을 먼저 보냅니다. 앞 부하나 캐시 상태가 창의 첫 요청들을 느리게 하지 않게
    # 모든 창에서 똑같이 보내고, 부하 전 스냅샷보다 앞이라 창의 값에는 들어가지 않습니다
    await clients[0].post(path, json={"question": questions[0]["question"]})

    # 부하 전 두 스냅샷과 시작 시각(UTC, Langfuse 기록을 읽을 때 씁니다)
    serving_before = read_metrics(SERVING_METRICS_URL, ADMIN_HEADERS)
    vllm_before = read_metrics(VLLM_METRICS_URL)
    load_start = datetime.now(timezone.utc)

    # 질문을 사용자 키에 번갈아 나눠 동시에 보냅니다(교안 01, 02 와 같은 방법)
    tasks = []
    for index, item in enumerate(questions):
        client = clients[index % len(clients)]
        tasks.append(run_limited(send_chat(client, path, {"question": item["question"]})))
    rows = pd.DataFrame(await asyncio.gather(*tasks))

    # 끝 시각과 부하 뒤 두 스냅샷
    load_end = datetime.now(timezone.utc)
    serving_after = read_metrics(SERVING_METRICS_URL, ADMIN_HEADERS)
    vllm_after = read_metrics(VLLM_METRICS_URL)

    return {
        "label": label,
        "path": path,
        "rows": rows,                     # send_chat 이 돌려준 요청별 행
        "load_start": load_start,
        "load_end": load_end,
        "serving_before": serving_before,
        "serving_after": serving_after,
        "vllm_before": vllm_before,
        "vllm_after": vllm_after,
    }

### 1-4. 되돌리기 워크플로 미리 켜기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 교안의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-deploy</code> 안에 있는 파일과 폴더 전체를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/serving-deploy/</code>에 덮어 복사합니다. 되돌리기 워크플로가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.github/workflows/rollback.yml</code>에 포함되어 있습니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
cd ~/serving-deploy


git add .
git commit -m "되돌리기 워크플로 켜기"
git push
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 push도 main push라 ci와 cd가 실행되고, 서버 코드는 그대로라 지금과 같은 서버가 이 커밋의 sha로 다시 배포됩니다. Actions 화면에서 ci와 cd가 모두 초록 체크가 되면 왼쪽 워크플로 목록에 rollback이 새로 보입니다. 3-1은 이 배포를 기준으로 삼습니다.</p>
</div>

## 2. SLI와 SLO

### 2-1. SLI, SLO, SLA

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><a style="color:#245fa0;text-decoration:underline;" href="https://sre.google/sre-book/service-level-objectives/">Google SRE 책의 SLO 장</a>과 <a style="color:#245fa0;text-decoration:underline;" href="https://sre.google/workbook/implementing-slos/">SRE workbook의 SLO 구현 장</a>이 정한 뜻을 그대로 씁니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">SLI(Service Level Indicator, 서비스 수준 지표):</strong> 서비스가 얼마나 잘 동작하는지 나타내는 숫자입니다. workbook처럼 좋은 이벤트 수를 유효 이벤트 수로 나눈 비율로 정합니다. 분모에 포함할 요청은 지표마다 정하며, 이 교안의 비율은 0과 1 사이이고 클수록 좋습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">SLO(Service Level Objective, 서비스 수준 목표):</strong> SLI의 목표값과 측정 창입니다. 측정 창 동안 SLI가 목표 이상이면 지킨 것입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">SLA(Service Level Agreement, 서비스 수준 협약):</strong> SLO를 지키지 못했을 때의 결과(환불, 위약금)를 사용자와 약속한 계약입니다. 이 교안은 계약 없이 SLO까지만 다룹니다.</li>
</ul>
</div>

<div class="day79-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">SLI</strong> = 좋은 요청 수 ÷ 유효 요청 수</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">예를 들어 한 창에서 유효 요청 64건 가운데 5xx가 아닌 응답이 63건이면 가용성 SLI는 63 / 64 = 0.984입니다. 가용성 SLO가 0.99이면 이 창은 목표를 지키지 못했습니다.</p>
</div>

### 2-2. 지연 SLI를 비율로 잡는 이유

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">지연 목표는 &quot;p95가 15초 이하&quot;처럼 분위수로 말할 수도 있고, &quot;15초 이하인 요청이 95% 이상&quot;처럼 비율로 말할 수도 있습니다. workbook은 지연 SLI를 기준보다 빠른 요청의 비율로 정합니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">히스토그램에서 정확함:</strong> 교안 01 6-2에서 본 것처럼 기준값이 칸 경계와 같으면 그 기준 이하 비율은 &quot;경계 이하 누적 건수 / 전체 건수&quot;로 정확히 나옵니다. 분위수는 어느 칸에 들었는지까지만 압니다. 예를 들어 경계가 10초와 15초인 히스토그램에서 &quot;15초 이하 비율&quot;은 바로 나오지만, p95가 그 두 경계 사이 칸에 들었다면 10초와 15초 사이 어디쯤이라고만 말할 수 있습니다. 서빙 API 히스토그램의 칸 경계에 15초를 넣은 이유입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">좋은 요청 수로 셀 수 있음:</strong> 비율이면 가용성, 품질과 같은 모양(좋은 / 유효)이 되어 에러 버짓(4절)을 같은 식으로 계산합니다.</li>
</ul>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/14_histogram_window.png" width="1050" alt="히스토그램 그림: 15초 경계 이하 비율은 누적 건수로 정확히 나오지만 p95는 칸 안 어딘가까지만 알 수 있음">

### 2-3. 이 서비스의 SLO 테이블

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">노무 창구의 SLO입니다. 지연 경계는 서빙 벤치마킹 단원의 지연 목표(TTFT 1,000ms, 달성률 0.95)와 교안 01의 서빙 API 15초 경계에서 가져왔습니다. TTFT(Time To First Token)는 요청이 들어온 뒤 첫 토큰이 나오기까지 걸린 시간입니다. 비용은 비율이 아니라 측정 창의 100만 출력 토큰당 운영 원가(교안 01 7절)에 상한을 둡니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적·활용:</strong> 지표별 목표와 좋은 요청 기준을 표로 정해 모든 측정 창을 같은 기준으로 판정합니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# 상담 창구의 SLO 테이블(이름, 목표 비율, 무엇이 좋은 요청인가, 어느 자료로 계산하나)
chat_slo = [
    {"이름": "가용성", "목표": 0.99, "좋은 요청": "5xx 가 아닌 응답", "자료": "서빙 API /metrics 의 serving_requests_total"},
    {"이름": "첫 토큰 지연", "목표": 0.95, "좋은 요청": "TTFT 1초 이하", "자료": "vLLM /metrics 의 vllm:time_to_first_token_seconds(경계 1.0)"},
    {"이름": "응답 지연", "목표": 0.95, "좋은 요청": "서빙 API 응답 15초 이하", "자료": "서빙 API /metrics 의 serving_request_seconds(경계 15.0)"},
    {"이름": "품질", "목표": 0.95, "좋은 요청": "답이 비지 않은 응답(200 응답 가운데 답 글자 수가 0 보다 큼)", "자료": "send_chat 이 받은 응답의 답 글자 수"},
]

# 비용 목표: 측정 창의 100만 출력 토큰당 운영 원가 상한(달러)
COST_SLO_PER_MILLION = 1.0

display(pd.DataFrame(chat_slo))

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">가용성의 유효 요청에서 401, 403, 422, 429를 빼는 이유</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">키가 없거나 틀림(401, 403), 요청 양식이 틀림(422), 요금제 한도를 넘김(429)은 서버가 정한 대로 거절한 응답입니다. 서비스가 잘못한 것이 아니므로 좋은 요청에도 나쁜 요청에도 넣지 않습니다. 넣으면 한 사용자가 한도를 넘길 때마다 가용성이 떨어집니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">좋은 요청을 세는 기준</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">품질의 좋은 요청:</strong> 200 응답 가운데 답이 비지 않은 응답입니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">send_chat</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">답 글자 수</code>가 0보다 큼). 답의 정확성이나 완결성까지 평가하지는 않습니다. 200 응답의 답이 모두 비지 않았다면 품질 SLI는 1.0이고, 생각이 생성 상한을 다 써 답이 비는 요청이 생기면 낮아집니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">지연의 예외:</strong> 서빙 API 히스토그램에는 상태 코드 라벨이 없어 429처럼 빨리 끝난 거절도 15초 이하로 셉니다. 이 교안의 부하는 한도 안에서 보내므로 거절이 없습니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 창</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실서비스:</strong> workbook은 4주 이동 창(rolling window, 지금부터 4주 전까지 계속 옮겨 가는 창)을 예로 듭니다. 요청이 많아 한두 건의 실패가 목표를 흔들지 않습니다. 4절의 경보 예는 workbook 경보 장을 따라 30일 창으로 계산합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">이 교안:</strong> 부하 한 번(노무 64건)이 창 하나이고, 창 하나가 목표를 지켰는지보다 기준 창과 다른 창의 차이를 봅니다.</li>
</ul>
</div>

### 2-4. SLI 계산 함수

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">기준 창과 장애 창에 같은 네 가지 계산을 적용해 비교하기 위해 함수로 묶습니다. 창의 수치는 부하 뒤에 읽은 누적값에서 부하 전에 읽은 누적값을 빼서 구합니다. 가용성과 응답 지연은 서빙 API, 첫 토큰 지연은 vLLM, 품질은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">send_chat</code>이 받은 요청별 결과를 씁니다.</p>
</div>

In [ ]:
# 히스토그램 경계(초). /metrics 의 le 라벨 값 그대로 문자열로 씁니다
E2E_EDGE = "15.0"    # 서빙 API serving_request_seconds 의 경계
TTFT_EDGE = "1.0"      # vLLM time_to_first_token_seconds 의 경계


def window_diff(window, layer, name, **labels):
    """서빙 API 또는 vLLM에서 한 측정 창 동안 늘어난 메트릭 값을 계산합니다.

    window는 measure_window가 돌려준 창이고, layer에는 "serving" 또는 "vllm"을 넣습니다.
    name은 메트릭 이름입니다. path나 status처럼 라벨을 주면 해당 라벨이 모두 맞는 숫자만 고릅니다.
    서버가 계속 누적하는 값에서 부하 전 숫자를 빼, 이번 부하에 해당하는 차이를 돌려줍니다.
    """
    after = metric_sum(window[layer + "_after"], name, **labels)
    before = metric_sum(window[layer + "_before"], name, **labels)
    return after - before

### 가용성에 쓸 응답 수 세기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">가용성에서는 401, 403, 422, 429 응답을 빼고 나머지 중 5xx가 아닌 응답을 셉니다.</p>
</div>

In [ ]:
def availability_counts(window):
    """측정 창의 가용성 SLI에 쓸 (좋은 응답 수, 유효 응답 수)를 계산합니다.

    서빙 API의 상태 코드별 요청 수를 씁니다. 401, 403, 422, 429는 사용자의 키나 요청 형식,
    요금제 한도 문제라 분모에서 빼고, 나머지 가운데 5xx가 아닌 응답을 좋은 응답으로 셉니다.
    """
    path = window["path"]
    statuses = []
    for row in window["serving_after"].itertuples():
        if row.name == "serving_requests_total" and row.labels["path"] == path:
            status = row.labels["status"]
            if status not in statuses:
                statuses.append(status)

    available_good = 0.0
    available_valid = 0.0
    for status in statuses:
        # 이 네 상태는 서비스가 잘못된 요청이나 한도를 거절한 경우라 유효 요청에서 뺍니다.
        if status in ["401", "403", "422", "429"]:
            continue
        count = window_diff(window, "serving", "serving_requests_total", path=path, status=status)
        available_valid += count
        if not status.startswith("5"):
            available_good += count
    return available_good, available_valid

### 네 가지 SLI 표 만들기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">가용성은 위에서 센 요청, 지연은 히스토그램, 품질은 응답의 답 글자 수를 사용합니다.</p>
</div>

In [ ]:
def slo_table(window):
    """측정 창의 SLI 네 개를 계산해 목표와 비교하는 표로 돌려줍니다.

    window에는 요청 경로, 요청별 결과, 서빙 API와 vLLM의 부하 전후 스냅샷이 들어 있습니다.
    각 행은 지표 하나이며, 좋은 수와 유효 수, SLI(좋은 수 / 유효 수), 목표, 남은 여유(SLI - 목표)를 담습니다.
    """
    path = window["path"]

    # 1. 가용성: 401·403·422·429를 뺀 응답 가운데 5xx가 아닌 응답
    available_good, available_valid = availability_counts(window)

    # 2. 첫 토큰 지연: TTFT 경계 이하 요청 / TTFT를 기록한 전체 요청
    ttft_good = window_diff(window, "vllm", "vllm:time_to_first_token_seconds_bucket", le=TTFT_EDGE)
    ttft_valid = window_diff(window, "vllm", "vllm:time_to_first_token_seconds_count")

    # 3. 응답 지연: 이 창구에서 15초 안에 끝난 요청 / 이 창구의 전체 요청
    e2e_good = window_diff(window, "serving", "serving_request_seconds_bucket", path=path, le=E2E_EDGE)
    e2e_valid = window_diff(window, "serving", "serving_request_seconds_count", path=path)

    # 4. 품질: 성공한 200 응답 가운데 답 글자 수가 0보다 큰 응답
    ok_rows = window["rows"][window["rows"]["상태 코드"] == 200]
    quality_good = (ok_rows["답 글자 수"] > 0).sum()
    quality_valid = len(ok_rows)

    # 지표 이름, 좋은 수, 유효 수를 모아 같은 방식으로 SLI와 목표 차이를 계산합니다.
    counts = [
        ("가용성", available_good, available_valid),
        ("첫 토큰 지연", ttft_good, ttft_valid),
        ("응답 지연", e2e_good, e2e_valid),
        ("품질", quality_good, quality_valid),
    ]
    targets = {}
    for rule in chat_slo:
        targets[rule["이름"]] = rule["목표"]

    rows = []
    for name, good, valid in counts:
        sli = good / valid
        target = targets[name]
        rows.append({"이름": name, "좋은 수": int(good), "유효 수": int(valid), "SLI": round(sli, 4),
                     "목표": target, "남은 여유": round(sli - target, 4)})
    return pd.DataFrame(rows)

### 출력 토큰 운영 원가 계산

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">창의 GPU 요금을 출력 토큰 수로 나눠 100만 토큰당 원가를 계산합니다.</p>
</div>

In [ ]:
def operating_cost(window):
    """측정 창의 100만 출력 토큰당 GPU 운영 원가(달러)를 계산합니다.

    창의 출력 토큰 수는 vLLM 누적값의 증가량입니다. GPU 비용은 시간당 요금에 창의 길이(초)를
    곱하고 3,600초로 나누어 계산합니다. GPU 비용을 출력 토큰 수로 나눈 뒤 1,000,000을 곱하고,
    화면에서 읽기 쉽도록 소수 셋째 자리까지 반올림해 돌려줍니다. 이 계산을 하려면 창에 출력 토큰이 있어야 합니다.
    """
    # 창의 출력 토큰은 vLLM 이 만든 토큰 수의 차이(교안 01 과 같은 자료)
    output_tokens = window_diff(window, "vllm", "vllm:generation_tokens_total")
    seconds = (window["load_end"] - window["load_start"]).total_seconds()
    gpu_cost = GPU_PRICE_PER_HOUR * seconds / 3600
    return round(gpu_cost / output_tokens * 1_000_000, 3)

## 3. 기준 측정과 판정

### 3-1. 기준 배포의 sha

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">지금 VM에서 도는 배포(교안 02의 코드에 1-4의 되돌리기 워크플로만 더한 커밋)를 기준으로 삼습니다. 6-1에서 배포 버전별 기록을 나누고, 7-2에서 이 sha로 되돌립니다.</p>
</div>

In [ ]:
# main push 의 ci 실행과 cd 실행을 읽습니다(맨 위가 가장 최근)
runs = recent_runs()
ci_runs = runs[(runs["name"] == "ci") & (runs["event"] == "push")]
cd_runs = runs[runs["name"] == "cd"]
# 맨 위 ci(이번 push)가 성공했고, 그 뒤에 생긴 cd(목록에서 그 ci 보다 위)도 성공했는지 확인합니다
assert ci_runs.iloc[0]["conclusion"] == "success", "이번 push 의 ci 가 아직 끝나지 않았거나 실패했습니다. Actions 화면을 보고 1분 뒤 다시 실행하세요."
assert cd_runs.index[0] < ci_runs.index[0] and cd_runs.iloc[0]["conclusion"] == "success", "이번 ci 에 이어진 cd 가 아직 성공하지 않았습니다. 1분 뒤 다시 실행하세요."

# 두 확인을 통과하면 맨 위 ci 의 커밋이 지금 VM 에서 도는 배포입니다
base_sha = ci_runs.iloc[0]["head_sha"]
print("기준 배포:", ci_runs.iloc[0]["display_title"], "| sha:", base_sha)

### 3-2. 기준 창 측정

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">노무 질문 64건을 동시 16으로 보내는 창 하나를 측정하고 SLO 테이블로 판정합니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 장애 배포 전의 SLI와 원가를 기준값으로 기록해, 장애·되돌리기·수정 뒤 결과와 비교합니다.</p>
</div>

In [ ]:
# 기준 창: 노무 질문 64건을 노무 키 셋에 나눠 보냅니다
base = await measure_window("기준", "/v1/labor/chat", labor_questions[:64], labor_clients)
display(base["rows"]["상태 코드"].value_counts())

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 기준 창이 정한 목표를 지켰는지 계산하고, 이후 창의 SLI·원가 비교에 사용할 표를 만듭니다.</p>
</div>

In [ ]:
# 기준 창의 SLI 테이블과 운영 원가
base_table = slo_table(base)
display(base_table)
print("100만 출력 토큰당 운영 원가(달러):", operating_cost(base), "| 목표 상한:", COST_SLO_PER_MILLION)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">목표에 못 미치는 값도 그대로 기준으로 둡니다. 원하는 값이 나올 때까지 다시 측정하면 비교 기준이 왜곡됩니다.
연결 오류로 측정이 중단되면 이미 보낸 요청의 처리가 끝나도록 1분쯤 기다린 뒤 다시 측정합니다. 오류가 반복되면 서버와 프록시의 연결 상태를 확인합니다.</p>
</div>

### 3-3. 전월세 기준 창(함께 따라하기 자료)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">8절 함께 따라하기가 쓸 전월세 창을 같은 배포에서 측정해 둡니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 전월세 기준값을 저장해, 뒤의 따라하기에서 장애 창과 비교합니다.</p>
</div>

In [ ]:
# 전월세 기준 창: 전월세 질문 32건을 전월세 키 둘에 나눠 보냅니다
lease_base = await measure_window("전월세 기준", "/v1/lease/chat", lease_questions[:32], lease_clients)
display(lease_base["rows"]["상태 코드"].value_counts())

## 4. 에러 버짓과 소진 속도

### 4-1. 에러 버짓

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">에러 버짓(error budget)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">SLO가 허용하는 나쁜 요청의 몫입니다. workbook의 정의는 100% - SLO이고, 목표 0.99이면 유효 요청의 1%까지 나빠도 됩니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 버짓이 남아 있으면 새 기능을 배포하고, 다 쓰면 배포를 멈추고 신뢰성을 고칩니다. 개발 속도와 안정성 가운데 무엇을 앞세울지를 숫자로 정합니다.</li>
</ul>
</div>

<div class="day79-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">에러 버짓</strong> = 1 − 목표<br><strong style="color:#1f3a5f;">허용 나쁜 수</strong> = (1 − 목표) × 유효 수<br><strong style="color:#1f3a5f;">남은 버짓</strong> = 허용 나쁜 수 − 나쁜 수</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">예를 들어 목표 0.99, 유효 요청 64건이면 허용 나쁜 수는 0.64건입니다. 한 건만 나빠도 이 창의 버짓을 넘깁니다. 4주 동안 유효 요청이 10만 건이면 같은 목표로 1,000건까지 나빠도 됩니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 목표가 허용한 나쁜 요청 수와 실제 나쁜 수의 차이를 계산해, 이 창에서 버짓을 남겼는지 넘겼는지 판단합니다.</p>
</div>

In [ ]:
# 기준 창의 허용 나쁜 수, 나쁜 수, 남은 버짓(요청 수)
base_table["허용 나쁜 수"] = ((1 - base_table["목표"]) * base_table["유효 수"]).round(2)
base_table["나쁜 수"] = base_table["유효 수"] - base_table["좋은 수"]
base_table["남은 버짓"] = (base_table["허용 나쁜 수"] - base_table["나쁜 수"]).round(2)
display(base_table)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">남은 버짓이 음수이면 이 창에서 허용한 나쁜 요청 수를 넘긴 것입니다.</p>
</div>

### 4-2. 소진 속도(burn rate)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">소진 속도(burn rate)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">SLO에 비해 에러 버짓을 얼마나 빨리 쓰는지를 나타내는 값입니다(<a style="color:#245fa0;text-decoration:underline;" href="https://sre.google/workbook/alerting-on-slos/">SRE workbook의 SLO 경보 장</a>). 나쁜 요청 비율이 버짓(1 - 목표)과 같으면 1이고, 이 속도로는 측정 창이 끝날 때 버짓을 딱 다 씁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 지금의 나쁜 요청 비율이 목표에서 허용한 비율보다 얼마나 큰지 비교해, 신뢰성을 고쳐야 할 때를 판단합니다. 그래서 경보를 이 값에 겁니다.</li>
</ul>
</div>

<div class="day79-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">소진 속도</strong> = 나쁜 요청 비율 ÷ (1 − 목표)</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">예를 들어 목표가 0.99(버짓 1%)인데 나쁜 요청 비율이 2%이면 소진 속도는 0.02 / 0.01 = 2입니다. 버짓을 정해진 속도의 두 배로 쓰는 상태입니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">workbook은 30일 SLO에 다음 경보를 예로 듭니다. 경보 창은 소진 속도를 계산하는 최근 시간 범위(예: 지난 1시간)입니다. 경보 창 동안 이 소진 속도가 이어지면 30일 버짓의 오른쪽 열만큼을 씁니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">경보 창</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">소진 속도</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">쓰는 30일 버짓</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">1시간</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">14.4</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">2%</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">6시간</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">6</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">5%</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">3일</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">1</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">10%</td>
</tr>
</tbody>
</table>
</div>

<div class="day79-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">쓰는 30일 버짓</strong> = 소진 속도 × 경보 창 ÷ 30일</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">첫 행: 14.4 × 1시간 ÷ 720시간 = 0.02, 곧 2%입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">짧은 창도 함께 봄:</strong> 실서비스는 지난 1시간 값과 함께 그 1/12인 지난 5분 값도 같은 기준을 넘을 때만 알립니다. 장애가 끝나면 5분 값이 먼저 내려가 경보가 곧 꺼집니다.</li>
</ul>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 나쁜 요청 비율이 목표 허용치보다 몇 배 빠른지 계산해, 경보·대응이 필요한 상태인지 판단합니다.</p>
</div>

In [ ]:
# 소진 속도 = 나쁜 요청 비율 / (1 - 목표)
base_table["소진 속도"] = ((1 - base_table["좋은 수"] / base_table["유효 수"]) / (1 - base_table["목표"])).round(2)
display(base_table[["이름", "SLI", "목표", "소진 속도"]])

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">소진 속도가 1보다 크면 목표가 허용한 비율보다 나쁜 요청이 많습니다.
64건 가운데 한 건이 나쁘고 목표가 0.99이면 소진 속도는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">(1 / 64) / 0.01 = 1.56</code>입니다. 요청이 적은 창에서는 한 건의 영향이 큽니다.</p>
</div>

## 5. 배포가 만든 장애

### 5-1. 장애 커밋 읽기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">&quot;모델이 오늘 날짜를 몰라 기한 계산을 틀린다&quot;는 사용자 의견을 받고, 시스템 메시지에 요청 시각을 알려 주는 작은 기능을 더한 커밋입니다. 커밋 메시지는 &quot;상담 답변에 요청 시각 알려 주기&quot;이고 바뀐 파일은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app/routers/metered_router.py</code> 하나입니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · 장애_커밋/serving-deploy/serving_app/routers/metered_router.py(더한 부분)</strong></div>

```python
# routers/metered_router.py
# 한국 시각(UTC+9)
KST = timezone(timedelta(hours=9))


def system_prompt_for(desk):
    """창구의 시스템 메시지 맨 앞에 지금 시각을 붙여 돌려준다(모델이 오늘 날짜를 알고 답하게)."""
    now = datetime.now(KST).strftime("%Y-%m-%d %H:%M:%S.%f")
    return f"[요청 시각] {now}\n\n" + SYSTEM_PROMPTS[desk]
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">상담 엔드포인트 두 곳(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">metered_chat</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">metered_chat_stream</code>)이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SYSTEM_PROMPTS[desk]</code> 대신 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">system_prompt_for(desk)</code>를 부릅니다.</li>
<li style="margin:3px 0;">배포 전 테스트는 상태 코드와 오류 양식만 보므로 이 커밋은 테스트를 통과하고 배포됩니다.</li>
</ul>
</div>

### 5-2. 커밋, push, 배포 확인

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">장애_커밋/serving-deploy</code> 안의 파일과 폴더 전체를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/serving-deploy/</code>에 덮어 복사합니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
cd ~/serving-deploy

git add .
git commit -m "상담 답변에 요청 시각 알려 주기"
git push
```

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GitHub 화면에서 할 일</strong></div>
<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day79-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/&lt;GitHub 아이디&gt;/serving-deploy/actions</code></li>
</ul>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">왼쪽 ci의 맨 위 실행에서 unit, image 두 상자가 초록 체크가 될 때까지 기다립니다.</li>
<li style="margin:3px 0;">이어서 시작되는 왼쪽 cd의 맨 위 실행에서 deploy 상자가 초록 체크가 될 때까지 기다립니다.</li>
</ol>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 이번 배포의 커밋 sha를 확인해 장애 창의 기록을 해당 이미지 버전에 연결합니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# 실행 목록을 다시 읽어 장애 커밋이 배포되었는지 봅니다(목록은 최대 60초 늦게 바뀝니다)
runs = recent_runs()
ci_runs = runs[(runs["name"] == "ci") & (runs["event"] == "push")]
cd_runs = runs[runs["name"] == "cd"]
# 맨 위 ci(이번 push)가 성공했고, 그 뒤에 생긴 cd(목록에서 그 ci 보다 위)도 성공했는지 확인합니다
assert ci_runs.iloc[0]["conclusion"] == "success", "이번 push 의 ci 가 아직 끝나지 않았거나 실패했습니다. Actions 화면을 보고 1분 뒤 다시 실행하세요."
assert cd_runs.index[0] < ci_runs.index[0] and cd_runs.iloc[0]["conclusion"] == "success", "이번 ci 에 이어진 cd 가 아직 성공하지 않았습니다. 1분 뒤 다시 실행하세요."
incident_sha = ci_runs.iloc[0]["head_sha"]
print("지금 배포:", ci_runs.iloc[0]["display_title"], "| sha:", incident_sha)

### 5-3. 장애 창 측정

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">기준 창과 같은 질문, 같은 키, 같은 동시 요청 수로 측정합니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 같은 부하를 장애 배포에서 다시 보내 SLI·원가 변화를 기준 창과 비교합니다.</p>
</div>

In [ ]:
# 장애 창: 기준 창과 같은 부하
incident = await measure_window("장애", "/v1/labor/chat", labor_questions[:64], labor_clients)
display(incident["rows"]["상태 코드"].value_counts())

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 장애 창의 SLI·소진 속도·원가를 계산해 기준값과 비교하고, 나빠진 지표를 찾습니다.</p>
</div>

In [ ]:
# 장애 창의 SLI 테이블, 소진 속도, 운영 원가
incident_table = slo_table(incident)
incident_table["소진 속도"] = ((1 - incident_table["좋은 수"] / incident_table["유효 수"]) / (1 - incident_table["목표"])).round(2)
display(incident_table)
print("100만 출력 토큰당 운영 원가(달러):", operating_cost(incident), "| 기준 창:", operating_cost(base))

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">기준 창보다 지연 SLI가 낮아졌는지 비교합니다. TTFT가 늘어도 1초 경계 안이면 SLI는 같을 수 있으므로 평균 TTFT와 운영 원가도 함께 봅니다.
가용성과 품질은 그대로인데 지연만 나빠졌다면, 실패 없이 느려지는 장애입니다.
소진 속도가 커질수록 목표가 허용한 비율보다 빠르게 버짓을 씁니다.</p>
<p style="margin:6px 0;">8절 함께 따라하기가 쓸 전월세 창도 같은 장애 배포에서 측정해 둡니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 전월세 장애 창을 저장해, 따라하기에서 같은 창구의 기준값과 비교합니다.</p>
</div>

In [ ]:
# 전월세 장애 창: 전월세 질문 32건을 전월세 키 둘에 나눠 보냅니다(8절 함께 따라하기 자료)
lease_incident = await measure_window("전월세 장애", "/v1/lease/chat", lease_questions[:32], lease_clients)
display(lease_incident["rows"]["상태 코드"].value_counts())

## 6. 원인 좁히기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">원인을 찾을 때는 나쁜 쪽만 보지 않고 나머지와 비교합니다. &quot;장애 창의 지연이 길다&quot;는 비교 대상이 없으면 원래 그런지, 이 배포 때문인지 말할 수 없습니다.</p>
<p style="margin:6px 0;">원인을 찾으면 SLO를 회복할 조치를 고릅니다. 이 교안에서는 이전 이미지로 되돌리기와 코드를 고친 커밋 두 가지를 비교합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/34_rca_loop.png" width="1050" alt="원인 분석 순서 그림: SLO로 감지, 나쁜 쪽과 나머지를 버전과 신호로 비교, 원인 가설, 대응 조치 적용, 같은 부하로 다시 측정">

### 6-1. 배포 버전별 trace 비교

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 02에서 trace에 붙인 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">version</code>(이미지 태그)으로 기준 창과 장애 창의 기록을 나눕니다. 기록은 두 창의 부하 시각 범위를 따로 읽어 합칩니다. 데움 요청과 3-3, 5-3의 전월세 창은 이 범위 밖이라 들어오지 않습니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 두 배포 버전의 요청별 지연을 비교해 장애 커밋에서 지연이 커졌는지 확인하고 원인 후보를 좁힙니다.</p>
</div>

In [ ]:
# 두 창의 성공 응답 수만큼 노무 기록이 들어올 때까지 30초마다 읽습니다(최대 10번, 교안 02 5-2 와 같은 모양)
success_count = int((base["rows"]["상태 코드"] == 200).sum() + (incident["rows"]["상태 코드"] == 200).sum())
for attempt in range(10):
    # 두 창의 부하 시각 범위를 따로 읽어 한 테이블로 합칩니다
    labor_generations = pd.concat([fetch_generations(base["load_start"], base["load_end"]),
                                   fetch_generations(incident["load_start"], incident["load_end"])])

    print(attempt + 1, "번째 읽기:", labor_generations["출력 토큰"].notna().sum(), "행 / 성공 응답", success_count, "건")
    if labor_generations["출력 토큰"].notna().sum() >= success_count:
        break
    time.sleep(30)

# 최대 10회 읽은 뒤에도 성공 응답 수만큼 기록이 없으면, 불완전한 표로 비교하지 않습니다
received_count = int(labor_generations["출력 토큰"].notna().sum())
if received_count < success_count:
    raise RuntimeError(f"Langfuse 기록이 아직 덜 도착했습니다: 성공 응답 {success_count}건 중 {received_count}건만 확인했습니다. 잠시 뒤 이 셀을 다시 실행하세요.")

# 배포 버전별 요청 수, 평균과 p95 지연, 15초 이하 비율
by_version = labor_generations.groupby("version")
version_table = by_version.agg(requests=("trace_id", "count"), mean_latency=("지연(초)", "mean"))
version_table["p95_latency"] = by_version["지연(초)"].quantile(0.95)
version_table["fast_ratio"] = (labor_generations["지연(초)"] <= 15).groupby(labor_generations["version"]).mean()

# 어느 버전이 어느 창인지 이름을 붙입니다
version_table["창"] = pd.Series({base_sha: "기준", incident_sha: "장애"})
display(version_table.round(3))

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">fast_ratio</code>는 모델 호출 시간으로 센 15초 이하 비율입니다. 서빙 API의 전체 응답 시간으로 센 응답 지연 SLI와 측정 범위가 다릅니다.</p>
<p style="margin:6px 0;">같은 질문과 동시 요청 수에서 장애 버전의 평균·p95 지연만 커졌다면, 해당 버전의 변경을 원인 후보로 봅니다.</p>
<p style="margin:6px 0;">같은 비교를 Langfuse 화면에서도 할 수 있습니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">Langfuse 화면에서 할 일</strong></div>
<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day79-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://cloud.langfuse.com/project/&lt;프로젝트 ID&gt;</code>(프로젝트 화면, 교안 02에서 연 trace 주소의 앞부분)</li>
</ul>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">왼쪽 Tracing을 열고 검색칸에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">version:&lt;sha&gt;</code>를 넣습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;sha&gt;</code>는 위 셀의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">incident_sha</code>입니다. Latency 열을 봅니다.</li>
<li style="margin:3px 0;">같은 검색에서 sha만 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">base_sha</code>로 바꿔 두 버전의 Latency를 비교합니다.</li>
</ol>
</div>

### 6-2. vLLM 신호와 대조

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서빙 API의 변경이 vLLM 안에서 무엇을 바꿨는지 vLLM 메트릭의 창 차이로 봅니다. 교안 01 5, 6절의 캐시 적중 비율과 TTFT에, 새로 대기 시간을 더해 창마다 계산하는 함수입니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">접두어 캐시 적중 비율:</strong> vLLM이 캐시 조회 대상으로 확인한 입력 토큰 가운데 캐시에서 재사용한 토큰의 비율입니다(교안 01 5절).</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">평균 대기 시간:</strong> 요청이 vLLM 대기열에서 처리를 기다린 시간(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm:request_queue_time_seconds</code>)의 평균입니다. 대기 요청 수 게이지는 순간 값이라, 창 전체는 이 히스토그램으로 봅니다.</li>
</ul>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 캐시 재사용, 첫 토큰 지연, 대기 시간을 창별로 계산해 API 지연 변화가 모델 안의 변화와 함께 나타났는지 살펴봅니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
def vllm_signals(window):
    """측정 창의 vLLM 메트릭으로 캐시 적중, 첫 토큰 지연, 대기 시간을 계산합니다.

    결과 딕셔너리에는 접두어 캐시 적중 비율, TTFT 경계 이하 비율, 평균 TTFT(초),
    평균 대기 시간(초)이 들어갑니다. 카운터와 히스토그램은 window_diff로 부하 중 증가한 양만 사용합니다.
    """
    signals = {"창": window["label"]}

    # 접두어 캐시 적중 비율 = 재사용한 토큰 수 차이 / 조회 대상 토큰 수 차이
    hits = window_diff(window, "vllm", "vllm:prefix_cache_hits_total")
    queries = window_diff(window, "vllm", "vllm:prefix_cache_queries_total")
    signals["접두어 캐시 적중 비율"] = round(hits / queries, 3)

    # TTFT: 경계 이하 비율과 평균(_sum 차이 / _count 차이)
    ttft_count = window_diff(window, "vllm", "vllm:time_to_first_token_seconds_count")
    signals["TTFT 1.0초 이하 비율"] = round(window_diff(window, "vllm", "vllm:time_to_first_token_seconds_bucket", le=TTFT_EDGE) / ttft_count, 3)
    signals["평균 TTFT(초)"] = round(window_diff(window, "vllm", "vllm:time_to_first_token_seconds_sum") / ttft_count, 3)

    # 대기열에서 기다린 시간의 평균
    queue_count = window_diff(window, "vllm", "vllm:request_queue_time_seconds_count")
    signals["평균 대기 시간(초)"] = round(window_diff(window, "vllm", "vllm:request_queue_time_seconds_sum") / queue_count, 3)
    return signals


signal_table = pd.DataFrame([vllm_signals(base), vllm_signals(incident)])
display(signal_table)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">장애 창에서 캐시 적중 비율이 낮아지고 평균 TTFT·대기 시간이 함께 늘었다면, 입력을 다시 계산하는 일이 늘어난 것으로 해석할 수 있습니다.</p>
</div>

### 6-3. 원인과 대응 방법

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">원인:</strong> 접두어 캐싱(서빙 벤치마킹 단원 교안 02)은 프롬프트 토큰을 맨 앞부터 정해진 토큰 수의 블록으로 나누고, 그 블록과 앞의 블록이 모두 같을 때만 계산해 둔 KV 캐시를 다시 씁니다. 장애 커밋은 마이크로초까지 붙인 요청 시각을 시스템 메시지 <strong style="font-size:1.13em;color:#1f3a5f;">맨 앞</strong>에 붙였으므로, 시각이 든 앞쪽 블록부터 요청마다 달라지고 그 뒤 블록은 글자가 같아도 모두 다시 계산합니다.</p>
<p style="margin:6px 0;">예를 들어 노무 질문 두 개가 1초 간격으로 들어온다고 합니다. 기준 배포에서는 두 요청의 시스템 메시지가 똑같아, 두 번째 요청이 첫 요청이 계산해 둔 지침과 참고 조문 블록을 그대로 씁니다. 장애 배포에서는 두 요청의 맨 앞 시각이 달라 시각이 든 블록부터 어긋나므로, 뒤의 지침과 참고 조문 블록은 글자가 같아도 처음부터 다시 계산합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/33_incident_prefix_cache.png" width="1050" alt="장애 원인 그림: 시스템 메시지 맨 앞에 요청 시각이 붙으면 시각이 든 블록부터 달라져 뒤의 지침과 참고 조문 블록을 다시 쓰지 못하고, 맨 뒤에 붙이면 앞 블록을 그대로 다시 씀">

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서빙 벤치마킹 단원에서 같은 노무 부하(L4 GPU, 동시 16)를 접두어 캐싱을 끈 서버와 켠 서버로 비교했을 때 평균 TTFT는 끔 2,892ms, 켬 280ms였습니다. 장애 커밋에서는 접두어 캐싱이 켜져 있어도 요청마다 시스템 메시지 맨 앞이 달라 공통된 뒤쪽 블록을 재사용하지 못합니다. 그래서 캐시를 끈 경우처럼 입력 토큰을 더 많이 계산하고 TTFT가 길어질 수 있습니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">대응 방법</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">적용에 걸리는 시간</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">기능 영향</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">되돌리기</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">rollback 워크플로(1-4에서 켜 두었습니다)로 기준 배포 sha의 이미지를 다시 띄웁니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">빌드 없이 배포(완료 시간은 대기·확인에 따라 다름)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">모델이 답변에 오늘 날짜를 반영할 수 없습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">고친 커밋</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">바뀌는 값을 시스템 메시지 맨 뒤로 옮기고, 초 대신 날짜만 붙입니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">ci(테스트, 빌드) 뒤 cd 배포</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">답변에 날짜를 반영하는 기능을 유지합니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">되돌리기는 코드를 고치지 않고 VM에서 도는 이미지만 기준 배포의 것으로 바꾸므로, 저장소 main에는 장애 커밋이 그대로 남습니다. 고친 커밋은 main의 코드를 고쳐 새 이미지를 만듭니다.</p>
<p style="margin:6px 0;">고친 커밋도 날짜가 바뀌면 시스템 메시지의 끝부분이 달라집니다. 날짜가 들어간 마지막 블록(들)은 다시 계산하지만, 그 앞의 동일한 완성 블록은 캐시에서 재사용할 수 있습니다. 요청마다 바뀌는 값(시각, 사용자 이름, 요청 id)을 프롬프트에 넣을 때는 이처럼 맨 뒤에 둡니다.</p>
</div>

## 7. 개선

### 7-1. 되돌리기 워크플로

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">배포한 뒤에 문제가 드러나면, 고친 커밋을 만들기 전에 먼저 서비스를 이전 상태로 돌려 놓아야 사용자의 실패와 버짓 소진이 늘지 않습니다. 실서비스라면 5-3에서 장애를 감지한 즉시 되돌리고 원인은 그 뒤에 찾습니다. 이 교안은 장애 창의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code> 스냅샷과 trace를 이미 받아 두었으므로 원인 분석을 6절에서 먼저 보였습니다. 1-4에서 켜 둔 되돌리기 워크플로를 읽고, 기준 배포의 이미지로 되돌립니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">되돌리기(rollback)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">실행 중인 배포를, 이전에 배포에 성공한 버전의 이미지로 바꾸는 일입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">sha 태그로 되돌리는 이유:</strong> 배포 단원의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ci.yml</code>은 테스트를 통과한 커밋마다 커밋 sha를 태그로 붙인 이미지를 GHCR에 올립니다. 그 태그로 compose를 다시 실행하면 다시 빌드하지 않고, 그때 테스트를 통과한 이미지를 그대로 띄웁니다. 이 교안에서는 GHCR에 기준 배포 sha의 이미지와 장애 커밋 sha의 이미지가 함께 있고, 기준 배포 sha를 골라 그 이미지를 다시 띄웁니다.</li>
<li style="margin:3px 0;">다음 main push가 배포되면 되돌린 상태는 끝납니다.</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/37_rollback_by_tag.png" width="1050" alt="되돌리기 그림: GHCR의 sha 태그 이미지 가운데 이전에 성공한 태그를 골라 VM의 컨테이너를 바꿈">
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · .github/workflows/rollback.yml</strong></div>

```yaml
# 이전 이미지로 되돌리는 워크플로(관측 단원 교안 03). 저장소의 .github/workflows 에 두면 Actions 화면에서 Run workflow 로 실행한다
name: rollback

on:
  workflow_dispatch:
    inputs:
      image_tag:
        description: "되돌릴 이미지 태그(ci 를 통과해 배포했던 커밋의 sha 전체)"
        required: true

permissions:
  contents: read

jobs:
  deploy:
    runs-on: ubuntu-latest
    timeout-minutes: 10
    concurrency:
      group: vm-deploy
      cancel-in-progress: false
    steps:
      - name: 저장소 받기
        uses: actions/checkout@v7
      - name: 지정한 이미지로 VM 배포하고 /healthz 확인
        env:
          VM_SSH_KEY: ${{ secrets.VM_SSH_KEY }}
          VM_HOST: ${{ vars.VM_HOST }}
          VM_USER: ${{ vars.VM_USER }}
          IMAGE_TAG: ${{ inputs.image_tag }}        # 입력값은 env 로만 넘긴다(run 줄에 바로 쓰지 않는다)
        run: bash scripts/deploy_vm.sh
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">workflow_dispatch</code>:</strong> 사람이 Run workflow 버튼을 눌러 실행하는 트리거입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">on:</code>에 이것만 있으므로 push로는 실행되지 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">inputs</code>:</strong> Run workflow 버튼을 누르면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">image_tag</code> 칸이 나오고, 입력한 값은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">inputs.image_tag</code>로 읽습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">입력값은 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">env</code>로만 넘기기:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run:</code> 줄에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&#36;{{ inputs.image_tag }}</code>를 바로 쓰면, GitHub가 값을 스크립트 글자에 끼워 넣은 뒤 셸이 실행하므로 입력한 글자가 명령으로 실행될 수 있습니다. 예를 들어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">abc; ls</code>를 입력하면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">;</code> 뒤의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ls</code>가 따로 실행됩니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">env:</code>로 넘기면 러너의 셸에는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">IMAGE_TAG</code> 변수의 값으로만 들어갑니다. VM에 보내는 명령에는 이 값이 다시 글자로 들어가므로, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">scripts/deploy_vm.sh</code>의 0) 단계가 커밋 sha 모양(16진수 40자리)만 통과시킵니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">ci 없이 deploy만:</strong> 되돌릴 이미지는 그 커밋의 ci 실행에서 이미 테스트를 통과했습니다. 배포는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cd.yml</code>과 같은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">deploy_vm.sh</code>로 하고, 같은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">concurrency</code> 그룹(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vm-deploy</code>)이라 두 배포가 겹쳐 실행되지 않습니다.</li>
</ul>
</div>

### 7-2. 되돌릴 sha 고르기와 되돌리기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">되돌릴 이미지 태그는 문제 배포 전 성능을 측정한 기준 배포의 sha입니다. 3-1에서 저장한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">base_sha</code>를 사용합니다.</p>
</div>

In [ ]:
# 되돌릴 이미지 태그는 3-1 의 기준 배포 sha 입니다(커밋 sha 는 비밀이 아니므로 출력합니다)
rollback_sha = base_sha
print("되돌릴 이미지 태그:", rollback_sha)
print("rollback 워크플로 화면:", REPO_URL + "/actions/workflows/rollback.yml")

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GitHub 화면에서 할 일</strong></div>
<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day79-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/&lt;GitHub 아이디&gt;/serving-deploy/actions/workflows/rollback.yml</code></li>
</ul>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">오른쪽 Run workflow를 누르고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">image_tag</code> 칸에 위 셀이 출력한 sha 40자를 붙여 넣은 뒤 초록 Run workflow 버튼을 누릅니다.</li>
<li style="margin:3px 0;">실행 목록에 rollback 실행이 생기고, deploy 상자 하나가 초록 체크가 될 때까지 기다립니다. 로그의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실행 중인 이미지:</code> 줄 끝이 입력한 sha입니다.</li>
</ol>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 되돌린 뒤 기준 창과 같은 부하를 보내 SLI와 운영 원가가 어떻게 바뀌는지 측정합니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# 되돌린 뒤 창: 기준 창과 같은 부하
rolled_back = await measure_window("되돌린 뒤", "/v1/labor/chat", labor_questions[:64], labor_clients)
display(slo_table(rolled_back))

### 7-3. 고친 커밋

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">바뀌는 값을 시스템 메시지 맨 뒤로 옮기고, 마이크로초까지 붙인 시각 대신 날짜만 붙입니다. 커밋 메시지는 &quot;날짜를 시스템 메시지 뒤로 옮기기&quot;입니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · 수정_커밋/serving-deploy/serving_app/routers/metered_router.py(고친 함수)</strong></div>

```python
# routers/metered_router.py
def system_prompt_for(desk):
    """창구의 시스템 메시지 맨 뒤에 오늘 날짜를 붙여 돌려준다(모델이 오늘 날짜를 알고 답하게).

    요청마다 바뀌는 글자는 맨 뒤에 둔다. 앞부분(지침과 참고 조문)이 요청마다 같아야 vLLM 의 접두어 캐시가 그대로 쓰인다.
    """
    today = datetime.now(KST).strftime("%Y-%m-%d")
    return SYSTEM_PROMPTS[desk] + f"\n\n[오늘 날짜] {today}"
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">수정_커밋/serving-deploy</code> 안의 파일과 폴더 전체를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/serving-deploy/</code>에 덮어 복사합니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
cd ~/serving-deploy

git add .
git commit -m "날짜를 시스템 메시지 뒤로 옮기기"
git push
```

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GitHub 화면에서 할 일</strong></div>
<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day79-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/&lt;GitHub 아이디&gt;/serving-deploy/actions</code></li>
</ul>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">왼쪽 ci의 맨 위 실행에서 unit, image 두 상자가 초록 체크가 될 때까지 기다립니다.</li>
<li style="margin:3px 0;">이어서 시작되는 왼쪽 cd의 맨 위 실행에서 deploy 상자가 초록 체크가 될 때까지 기다립니다. 이 배포가 되돌린 상태를 끝냅니다.</li>
</ol>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 고친 커밋이 배포된 뒤 기준 창과 같은 부하를 다시 보내 SLI와 운영 원가를 비교합니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# 수정 뒤 창: 기준 창과 같은 부하
fixed = await measure_window("수정 뒤", "/v1/labor/chat", labor_questions[:64], labor_clients)
display(slo_table(fixed))

### 7-4. 네 창 비교

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">같은 질문, 같은 키, 같은 동시 요청 수로 측정한 네 창을 같은 열의 테이블 하나로 놓습니다. 맨 위 행은 SLO 테이블의 목표입니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 목표·기준·장애·되돌린 뒤·수정 뒤 값을 나란히 놓고, 목표 회복 여부와 남은 차이를 판단합니다.</p>
</div>

In [ ]:
# 첫 행: SLO 테이블의 목표
target_row = {"창": "목표"}
for slo in chat_slo:
    target_row[slo["이름"]] = slo["목표"]
target_row["운영 원가(100만 출력 토큰당 달러)"] = COST_SLO_PER_MILLION
compare_rows = [target_row]

# 창마다 SLI 네 개, 운영 원가, 접두어 캐시 적중 비율, 참고 열을 한 행으로 모읍니다
for window in [base, incident, rolled_back, fixed]:
    row = {"창": window["label"]}
    for record in slo_table(window).to_dict("records"):
        row[record["이름"]] = record["SLI"]
    row["운영 원가(100만 출력 토큰당 달러)"] = operating_cost(window)
    row["접두어 캐시 적중 비율"] = vllm_signals(window)["접두어 캐시 적중 비율"]

    # 참고: 200 응답 가운데 끝난 이유가 length(생성 상한에서 끊김)인 비율. 판정에는 쓰지 않습니다
    ok_rows = window["rows"][window["rows"]["상태 코드"] == 200]
    row["참고: length 비율"] = round((ok_rows["끝난 이유"] == "length").mean(), 3)
    compare_rows.append(row)

compare_table = pd.DataFrame(compare_rows)
display(compare_table)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">장애 창의 캐시 적중 비율이 낮아지고 운영 원가가 높아졌는지 봅니다. 되돌린 뒤와 수정 뒤의 지연·캐시 적중 비율이 기준과 비슷해졌다면 성능이 회복된 것입니다.
수정 뒤에도 목표에 못 미치는 지표가 남으면 그 지표의 원인을 따로 확인합니다.</p>
</div>

## 8. 🖐️ 함께 따라하기: 전월세 창구의 SLO 판정과 소진 속도

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">본실습에서 노무 창구로 계산한 소진 속도를 전월세 창구에도 적용합니다. 새로 배포하거나 부하를 보내지 않고, 앞에서 측정한 두 창을 사용합니다.</p>
<p style="margin:6px 0;">두 창의 SLI 표는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">slo_table</code>로 만듭니다. 빈칸에 나쁜 요청 비율을 허용 나쁜 요청 비율로 나누는 식을 넣어 소진 속도를 비교합니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 목적·활용:</strong> 전월세 기준·장애 창의 SLI와 소진 속도를 비교해, 창구별로 목표를 얼마나 빠르게 쓰는지 확인합니다.</p>
</div>

In [ ]:
for window in [lease_base, lease_incident]:
    # 2-4의 함수를 다시 사용해 전월세 창의 SLI 표를 만듭니다.
    table = slo_table(window)

    # 에러 버짓을 쓰는 속도 = 나쁜 요청 비율 / 허용 나쁜 요청 비율
    bad_ratio = 1 - table["좋은 수"] / table["유효 수"]
    allowed_bad_ratio = 1 - table["목표"]
    table["소진 속도"] = ...

    print(window["label"], "| 100만 출력 토큰당 운영 원가(달러):", operating_cost(window))
    display(table)

## 9. 핵심 정리

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">SLI와 SLO:</strong> SLI = 좋은 요청 / 유효 요청, SLO = SLI의 목표와 측정 창입니다. 가용성은 401, 403, 422, 429를 제외하고, 응답 지연은 히스토그램의 요청 전체, 품질은 200 응답을 분모로 씁니다. 목표는 가용성 0.99, 첫 토큰 지연 0.95(TTFT 1.0초 이하), 응답 지연 0.95(15.0초 이하), 품질 0.95(답이 비지 않음), 100만 출력 토큰당 운영 원가 상한 1.0달러입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">측정 창:</strong> 부하 앞뒤의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code> 스냅샷 차이로 창 하나의 값을 계산합니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">measure_window</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">slo_table</code>). 실서비스에서는 4주 이동 창처럼 더 긴 범위를 정해 봅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">에러 버짓과 소진 속도:</strong> 버짓 = 1 - 목표, 소진 속도 = 나쁜 요청 비율 / (1 - 목표)입니다. workbook의 경보 예는 1시간 14.4(30일 버짓 2%), 6시간 6(5%), 3일 1(10%)이고, 짧은 창(긴 창의 1/12)도 함께 넘을 때 알립니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">원인 좁히기:</strong> 나쁜 쪽과 나머지를 비교합니다. trace의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">version</code>으로 배포 버전별 지연을 나누고, vLLM의 접두어 캐시 적중 비율, TTFT, 대기 시간으로 서버 안에서 무엇이 바뀌었는지 봅니다. 시스템 메시지 맨 앞에 요청마다 바뀌는 글자를 넣으면 접두어 캐시를 다시 쓰지 못합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">되돌리기 워크플로:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">rollback.yml</code>은 Run workflow의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">image_tag</code>(커밋 sha)를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">env</code>로만 넘겨, 빌드와 테스트 없이 그 sha의 이미지를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">deploy_vm.sh</code>로 다시 띄웁니다. 이 실습에서는 문제 배포 전 성능을 측정한 기준 배포로 돌아갑니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">개선:</strong> 되돌리기(rollback 워크플로, 빠름, 기능 잃음)로 먼저 서비스를 살리고, 바뀌는 값을 맨 뒤로 옮긴 고친 커밋(기능 유지)을 배포한 뒤, 같은 부하와 같은 테이블로 네 창을 비교합니다.</li>
</ul>
<p style="margin:6px 0;">교안 03을 마쳤으면 RunPod 콘솔에서 GPU Pod를 <strong style="font-size:1.13em;color:#1f3a5f;">Stop → Terminate</strong> 순서로 종료합니다.</p>
</div>